In [0]:
# Live 4 (O) — Orquestração
# Notebook: 11_silver_vendas_itens_pedido
# Objetivo: ler Bronze (parquet) e gerar a Silver de itens com deduplicação.

from pyspark.sql import functions as F
from pyspark.sql import types as T

dbutils.widgets.text("tgt_schema", "aulas_ao_vivo.live_20260323")
dbutils.widgets.text("process_date", "2026-03-01")

TGT_SCHEMA = dbutils.widgets.get("tgt_schema")
PROCESS_DATE = dbutils.widgets.get("process_date")

VOLUME_PATH = f"/Volumes/{TGT_SCHEMA.replace('.', '/')}/files"
BASE_PATH = f"{VOLUME_PATH}/live_20260323"
BRONZE_BASE_PATH = f"{BASE_PATH}/bronze"
PARQUET_PATH = f"{BRONZE_BASE_PATH}/ecommerce/pedidos_parquet/dt={PROCESS_DATE}"

TBL_SILVER_ITENS = f"{TGT_SCHEMA}.silver_vendas_itens_pedido"
TBL_DIM_PRODUTOS = f"{TGT_SCHEMA}.silver_catalogo_produtos"  # view criada no notebook 10

def standardize_column_names(df):
    import re
    cols = df.columns
    new_cols = []
    for c in cols:
        c2 = c.strip().lower()
        c2 = re.sub(r"[^a-z0-9]+", "_", c2)
        c2 = re.sub(r"_+", "_", c2).strip("_")
        new_cols.append(c2)
    out = df
    for old, new in zip(cols, new_cols):
        if old != new:
            out = out.withColumnRenamed(old, new)
    return out

"""
Fidelidade (Live 2):
- mesma padronização + status {pago,cancelado,pendente}
- parse de data com try_to_timestamp
- regras de null/blank em quantidade/produto
- dedup por (id_pedido, produto, data_pedido)
"""

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TGT_SCHEMA}")

bronze = spark.read.format("parquet").load(PARQUET_PATH)
print("Linhas lidas do bronze(parquet):", bronze.count())

vendas_1 = standardize_column_names(bronze)

rename_map = {"qtd": "quantidade", "valor_total_r": "valor_total"}
vendas_2 = vendas_1
for old, new in rename_map.items():
    if old in vendas_2.columns:
        vendas_2 = vendas_2.withColumnRenamed(old, new)

# Trim básico
v = vendas_2
for c, t in v.dtypes:
    if t == "string":
        v = v.withColumn(c, F.trim(F.col(c)))

v = (
    v
    .withColumn("id_pedido", F.upper(F.col("id_pedido")))
    .withColumn("uf", F.upper(F.col("uf")))
    .withColumn("produto", F.lower(F.col("produto")))
    .withColumn("status", F.lower(F.col("status")))
)

v = v.withColumn(
    "status",
    F.when(F.col("status").rlike("cancel"), F.lit("cancelado"))
     .when(F.col("status").rlike("pend"), F.lit("pendente"))
     .otherwise(F.lit("pago"))
)

v = v.withColumn(
    "data_pedido",
    F.coalesce(
        F.try_to_timestamp("data_pedido", F.lit("yyyy-MM-dd HH:mm:ss")),
        F.try_to_timestamp("data_pedido", F.lit("dd/MM/yyyy HH:mm:ss")),
        F.try_to_timestamp(F.concat(F.col("data_pedido"), F.lit(":00")), F.lit("dd/MM/yyyy HH:mm:ss"))
    )
)

v = v.withColumn(
    "quantidade",
    F.when((F.col("quantidade").isNull()) | (F.col("quantidade") == "") | (F.col("quantidade").rlike(r"^\s+$")), None)
     .otherwise(F.col("quantidade"))
     .cast("int")
)

v = v.withColumn(
    "produto",
    F.when((F.col("produto").isNull()) | (F.col("produto") == "") | (F.col("produto").rlike(r"^\s+$")), None)
     .otherwise(F.col("produto"))
)

valor_raw = F.col("valor_total")
valor_sem_ponto_milhar = F.when(valor_raw.contains(","), F.regexp_replace(valor_raw, "\\.", "")).otherwise(valor_raw)
valor_ponto_decimal = F.regexp_replace(valor_sem_ponto_milhar, ",", ".")
valor_sem_espacos = F.regexp_replace(valor_ponto_decimal, " ", "")

v = (
    v
    .withColumn("valor_total", valor_sem_espacos.cast(T.DecimalType(10, 2)))
    .withColumn("dt", F.to_date(F.col("data_pedido")))
    .withColumn("mes_referencia", F.trunc(F.col("data_pedido"), "MM"))
)

pre = v.filter(
    F.col("produto").isNotNull() &
    F.col("quantidade").isNotNull() &
    F.col("data_pedido").isNotNull() &
    F.col("valor_total").isNotNull()
)

silver_itens = pre.dropDuplicates(["id_pedido", "produto", "data_pedido"]) 
print("Linhas antes dedup:", pre.count())
print("Linhas depois dedup:", silver_itens.count())

# Checagem didática: produtos em itens sem cadastro na dimensão
dim = spark.table(TBL_DIM_PRODUTOS)
produtos_sem_cadastro = (
    silver_itens.select("produto").distinct()
    .join(dim.select("produto").distinct(), on="produto", how="left_anti")
)
print("Produtos sem cadastro (deve ser vazio se catálogo cobre tudo):")
display(produtos_sem_cadastro)

silver_itens.write.mode("overwrite").format("delta").saveAsTable(TBL_SILVER_ITENS)

out = spark.table(TBL_SILVER_ITENS)
out.printSchema()
display(out.orderBy(F.col("data_pedido").asc()).limit(50))
